In [0]:
dbutils.secrets.list('storage_key')

In [0]:
storage_account = "azurestoragepramod"
storage_key = dbutils.secrets.get(
    scope="storage_key",
    key="storage-key"
)
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)

In [0]:
dbutils.widgets.text("pipeline_run_id",'')
dbutils.widgets.text("pipline_name",'')
dbutils.widgets.text("source_name",'')
dbutils.widgets.text("file_name",'')
dbutils.widgets.text("landing_area_path",'')
dbutils.widgets.text("pipline_status", "")
dbutils.widgets.text("pipline_start_time", "")
dbutils.widgets.text("pipline_end_time", "")
dbutils.widgets.text("pipeline_error_message", "")
dbutils.widgets.text("schema_validation_status","")
dbutils.widgets.text("dup_validation_status","")
dbutils.widgets.text("datatype_mismatch","")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
pipline_name = dbutils.widgets.get("pipline_name")
source_name = dbutils.widgets.get("source_name")
file_name = dbutils.widgets.get("file_name")
landing_area_path= dbutils.widgets.get("landing_area_path")
pipline_status = dbutils.widgets.get("pipline_status")
pipline_start_time = dbutils.widgets.get("pipline_start_time")
pipline_end_time = dbutils.widgets.get("pipline_end_time")
pipeline_error_message = dbutils.widgets.get("pipeline_error_message")
schema_validation_status=dbutils.widgets.get("schema_validation_status")
dup_validation_status=dbutils.widgets.get("dup_validation_status")
datatype_mismatch=dbutils.widgets.get("datatype_mismatch")

In [0]:
%sql
create schema if not exists control;

In [0]:
%sql

create table if not exists pramoddatabricks.control.bronze_audit(
    pipeline_run_id string,
    pipline_name string,
    source_name string,
    file_name string,
    landing_area_path string,
    bronze_table_name string,
    source_count string,
    target_count string,
    schema_status string, 
    overall_status string,
    message string,
    pipline_start_time timestamp,
    pipline_end_time timestamp,
    pipline_status string,
    pipeline_error_message string
);



In [0]:
from pyspark.sql import Row, functions
from pyspark.sql.functions import *

if pipline_status == "Success":
    df = spark.createDataFrame([
        Row(
            pipline_name=pipline_name,
            source_name=source_name,
            file_name=file_name,
            landing_area_path=landing_area_path,
            bronze_table_name=f"pramoddatabricks.bronze.{source_name}",
            source_count="Not validated",
            target_count="Not validated",
            schema_status="Success",
            overall_status="Success",
            message="Success",
            pipline_start_time=pipline_start_time,
            pipline_end_time=pipline_end_time,
            pipline_status=pipline_status,
            pipeline_error_message="Success",
            pipline_run_id=pipeline_run_id
        )
    ])
else:
    df = spark.createDataFrame([
        Row(
            pipline_name=pipline_name,
            source_name=source_name,
            file_name=file_name,
            landing_area_path=landing_area_path,
            bronze_table_name=f"pramoddatabricks.bronze.{source_name}",
            source_count="Not validated",
            target_count="Not validated",
            schema_status="Failed",
            overall_status="Failed",
            message="Failed",
            pipline_start_time=pipline_start_time,
            pipline_end_time=pipline_end_time,
            pipline_status=pipline_status,
            pipeline_error_message="Failed",
            pipline_run_id=pipeline_run_id
        )
    ])

from pyspark.sql.functions import col, to_timestamp

df = (
    df.withColumn(
        "pipline_start_time",
        to_timestamp(col("pipline_start_time"))
    )
    .withColumn(
        "pipline_end_time",
        to_timestamp(col("pipline_end_time"))
    ))

df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("pramoddatabricks.control.bronze_audit")

In [0]:
%sql
select * from pramoddatabricks.control.bronze_audit